# 2단계: threshold × NMS 후처리 최적화

이 노트북은 `monai_mini` 체크포인트를 대표 모델로 고정해, 데이터셋 × 타겟 방식의 네 조합에서 val 기반 후처리 탐색을 재현한다.

- 탐색 격자: `threshold ∈ {0.2, 0.3, 0.4, 0.5, 0.6}` × `nms_radius ∈ {5, 10, 15, 20, 30}`
- 선택 기준: 1단계의 **헝가리안 F1@50 px**. 동률이면 recall, precision, 더 작은 threshold, 더 작은 NMS 반경 순으로 결정한다.
- 누수 방지: 격자 전체는 val에서만 선택하고, 선택된 조합 하나만 test에서 재평가한다.
- 비용 절감: 모델 추론은 조합별 val에서 한 번만 수행하고, 같은 히트맵에 25개 후처리를 적용한다.

`monai_mini`를 고정한 이유는 개선 계획의 완료 조건이 네 개(데이터셋 × 타겟) 최적점을 요구하고, 1단계 보고서가 모델 크기 축의 효과가 작다고 결론내렸기 때문이다. 다른 모델 타입의 최적화는 이 노트북의 범위 밖이다.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import subprocess
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from IPython.display import Image as DisplayImage, Markdown, display
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader

for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (candidate / "ttd").is_dir():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Run the notebook from this repository or its notebook/ directory.")
sys.path.insert(0, str(REPO_ROOT))

from ttd.dataset import SurgicalToolDataset, require_samples
from ttd.evaluation import hungarian_match
from ttd.model import build as build_model
from ttd.peaks import find_peaks
from ttd.transforms import _eval_transform

THRESHOLDS = (0.2, 0.3, 0.4, 0.5, 0.6)
NMS_RADII = (5, 10, 15, 20, 30)
MATCH_DISTANCE_PX = 50.0
MODEL_TYPE = "monai_mini"
BATCH_SIZE = 16
WORKERS = 0
DEVICE_BY_COMBINATION = {
    ("erop", "gradient-seg"): "cuda:0",
    ("erop", "gaussian-tip"): "cuda:1",
    ("cholec80", "gradient-seg"): "cuda:2",
    ("cholec80", "gaussian-tip"): "cuda:3",
}
SWEEP_ROOT = REPO_ROOT / "data/results/phase2/val-grid"
TEST_ROOT = REPO_ROOT / "data/results/phase2/test"
SNAPSHOT_PATH = REPO_ROOT / "data/dataset-snapshot-2026-08-20.txt"

assert SNAPSHOT_PATH.is_file(), "0단계 데이터셋 스냅샷이 필요합니다."
assert torch.cuda.is_available(), "이 실험은 CUDA GPU를 전제로 합니다."
print(f"PyTorch: {torch.__version__}; CUDA: {torch.cuda.get_device_name(0)}")

PyTorch: 2.7.1+cu128; CUDA: NVIDIA RTX A6000


In [2]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_gt(annotation_path: str) -> list[tuple[float, float]]:
    with open(annotation_path, encoding="utf-8") as handle:
        return [
            (annotation["tip"]["x"], annotation["tip"]["y"])
            for annotation in json.load(handle)["annotations"]
        ]


def nms(peaks: list[tuple[int, int, float]], radius: int) -> list[tuple[int, int, float]]:
    """Same deterministic NMS used by ttd.peaks.find_peaks()."""
    kept = []
    for peak in peaks:
        if all(np.hypot(peak[0] - other[0], peak[1] - other[1]) >= radius for other in kept):
            kept.append(peak)
    return kept


def metric(tp: int, fp: int, fn: int) -> dict:
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": 2 * precision * recall / max(1e-12, precision + recall),
    }


def sweep_val(dataset_name: str, target_mode: str, device_name: str) -> dict:
    """Run one val forward pass and score all 25 post-processing settings."""
    data_root = REPO_ROOT / "data/dataset" / dataset_name
    model_path = REPO_ROOT / "data/models" / dataset_name / target_mode / MODEL_TYPE / "best.pt"
    device = torch.device(device_name)
    dataset = SurgicalToolDataset(str(data_root), "val", transform=_eval_transform(), target_mode=target_mode)
    require_samples(dataset, "val", str(data_root))
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=WORKERS,
                        pin_memory=device.type == "cuda")
    model = build_model(MODEL_TYPE, num_classes=2).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=False))
    model.eval()

    counts = {(threshold, radius): {"tp": 0, "fp": 0, "fn": 0}
              for threshold in THRESHOLDS for radius in NMS_RADII}
    started = time.time()
    with torch.no_grad():
        for batch_index, (images, _) in enumerate(loader):
            heatmaps = torch.sigmoid(model(images.to(device, dtype=torch.float32))[:, 1]).cpu().numpy()
            for offset, heatmap in enumerate(heatmaps):
                annotation_path = dataset.samples[batch_index * BATCH_SIZE + offset]
                gt_tips = read_gt(annotation_path)
                # Compute connected-component maxima once per threshold.  NMS is
                # then reapplied for every radius, exactly matching find_peaks().
                peaks_by_threshold = {threshold: find_peaks(heatmap, threshold, 0)
                                      for threshold in THRESHOLDS}
                for threshold, preliminary_peaks in peaks_by_threshold.items():
                    for radius in NMS_RADII:
                        candidates = nms(preliminary_peaks, radius)
                        matches = hungarian_match(gt_tips, candidates, MATCH_DISTANCE_PX)
                        tp = len(matches)
                        counts[(threshold, radius)]["tp"] += tp
                        counts[(threshold, radius)]["fn"] += len(gt_tips) - tp
                        counts[(threshold, radius)]["fp"] += len(candidates) - tp
            done = min((batch_index + 1) * BATCH_SIZE, len(dataset))
            if done % 5000 < BATCH_SIZE or done == len(dataset):
                print(f"{dataset_name}/{target_mode}: {done:,}/{len(dataset):,}")

    records = []
    for threshold in THRESHOLDS:
        for radius in NMS_RADII:
            result = metric(**counts[(threshold, radius)])
            records.append({"threshold": threshold, "nms_radius": radius, **result})
    selected = max(records, key=lambda row: (row["f1"], row["recall"], row["precision"], -row["threshold"], -row["nms_radius"]))
    return {
        "schema_version": 1,
        "dataset": dataset_name,
        "target_mode": target_mode,
        "model_type": MODEL_TYPE,
        "model_path": str(model_path.relative_to(REPO_ROOT)),
        "model_sha256": sha256(model_path),
        "split": "val",
        "dataset_snapshot": str(SNAPSHOT_PATH.relative_to(REPO_ROOT)),
        "match_distance_px": MATCH_DISTANCE_PX,
        "thresholds": list(THRESHOLDS),
        "nms_radii": list(NMS_RADII),
        "selection_rule": "max F1; then recall, precision, lower threshold, lower NMS radius",
        "records": records,
        "selected": selected,
        "elapsed_seconds": round(time.time() - started, 2),
        "device": device_name,
    }


def save_sweep(result: dict) -> Path:
    output = SWEEP_ROOT / result["dataset"] / result["target_mode"] / f"{MODEL_TYPE}.json"
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    return output

In [ ]:
# Run all four val sweeps.  Re-running overwrites only the deterministic phase-2
# val-grid JSON files; it never writes test metrics or model checkpoints.
results = {}
for dataset_name in ("erop", "cholec80"):
    for target_mode in ("gradient-seg", "gaussian-tip"):
        key = (dataset_name, target_mode)
        result = sweep_val(*key, DEVICE_BY_COMBINATION[key])
        output = save_sweep(result)
        results[key] = result
        print(f"selected {key}: threshold={result['selected']['threshold']}, "
              f"nms={result['selected']['nms_radius']}, F1={result['selected']['f1']:.4f} → {output}")

erop/gradient-seg: 5,008/36,140
erop/gradient-seg: 10,000/36,140
erop/gradient-seg: 15,008/36,140
erop/gradient-seg: 20,000/36,140


In [ ]:
def render_f1_heatmap(result: dict) -> Path:
    cell = 120
    left, top = 120, 90
    width, height = left + cell * len(NMS_RADII), top + cell * len(THRESHOLDS) + 40
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    values = {(row["threshold"], row["nms_radius"]): row["f1"] for row in result["records"]}
    low, high = min(values.values()), max(values.values())
    for col, radius in enumerate(NMS_RADII):
        draw.text((left + col * cell + 42, 30), str(radius), fill="black")
    for row, threshold in enumerate(THRESHOLDS):
        draw.text((25, top + row * cell + 45), f"{threshold:.1f}", fill="black")
        for col, radius in enumerate(NMS_RADII):
            value = values[(threshold, radius)]
            fraction = 0.5 if high == low else (value - low) / (high - low)
            color = (int(255 * (1 - fraction)), int(235 * (1 - fraction)), int(255 * fraction))
            x, y = left + col * cell, top + row * cell
            draw.rectangle((x, y, x + cell - 2, y + cell - 2), fill=color, outline="black")
            draw.text((x + 30, y + 48), f"{value:.3f}", fill="black")
    draw.text((left, 5), "NMS radius (px)", fill="black")
    draw.text((5, 65), "threshold", fill="black")
    output = SWEEP_ROOT / result["dataset"] / result["target_mode"] / f"{MODEL_TYPE}-f1-heatmap.png"
    image.save(output)
    return output

summary_rows = []
for key, result in results.items():
    selected = result["selected"]
    summary_rows.append(f"| `{key[0]}` | `{key[1]}` | {selected['threshold']:.1f} | {selected['nms_radius']} px | {selected['precision']:.3f} | {selected['recall']:.3f} | **{selected['f1']:.3f}** |")
    display(Markdown(f"### {key[0]} / {key[1]}"))
    display(DisplayImage(filename=str(render_f1_heatmap(result))))
display(Markdown("""### Val-selected settings

| dataset | target | threshold | NMS radius | precision | recall | F1 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
""" + "\n".join(summary_rows)))

## Held-out test confirmation

이 셀은 위 val 선택값 하나만 각 조합의 test에 적용한다. 결과는 기존 1단계 기준선(`data/results/<dataset>/...`)을 덮어쓰지 않고 `data/results/phase2/test/` 아래에 저장한다. 이 결과로 기존 단일 설정과의 개선폭, 공유 피크 비율의 변화, 연결요소 분할 착수 여부를 판단한다.

In [ ]:
# Execute only after the val selection cell has completed successfully.
for (dataset_name, target_mode), result in results.items():
    selected = result["selected"]
    output = TEST_ROOT / dataset_name / target_mode / MODEL_TYPE
    command = [
        sys.executable, "scripts/eval-model.py",
        "--dataset", dataset_name,
        "--target-mode", target_mode,
        "--model-type", MODEL_TYPE,
        "--threshold", str(selected["threshold"]),
        "--nms-radius", str(selected["nms_radius"]),
        "--results-dir", str(output),
        "--apply-bias",
        "--device", DEVICE_BY_COMBINATION[(dataset_name, target_mode)],
        "--workers", str(WORKERS),
        "--batch-size", str(BATCH_SIZE),
    ]
    print(" ".join(command))
    subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
# Summarise the held-out test results and compare them with the 1-stage baseline.
for (dataset_name, target_mode), result in results.items():
    baseline_path = REPO_ROOT / "data/results" / dataset_name / target_mode / MODEL_TYPE / "summary.json"
    optimized_path = TEST_ROOT / dataset_name / target_mode / MODEL_TYPE / "summary.json"
    baseline = json.loads(baseline_path.read_text())
    optimized = json.loads(optimized_path.read_text())
    base = baseline["hungarian"]["distance_caps_px"]["50"]
    tuned = optimized["hungarian"]["distance_caps_px"]["50"]
    print(f"{dataset_name}/{target_mode}: "
          f"F1@50 {base['f1']:.3f} → {tuned['f1']:.3f} ({tuned['f1'] - base['f1']:+.3f}), "
          f"recall {base['recall']:.3f} → {tuned['recall']:.3f}")
